# Auditing and Editing
## Overview 

Howso Engine's instance-based machine learning approach enables unique capabilities in addition to interpretability, which we learned about in the other recipes. We were able to detect possible anomalies and investigate Influential Cases and features that may be of concern. Using that information combined with Howso Engine giving us dynamic control over our Trainee, we can take meaningful action without having to dramatically incur additional expenses such as recreating the model. This is in contrast to most machine learning models which, once trained, are difficult to update without retraining the entire model. 

In this notebook we demonstrate the editability of a Howso Engine Trainee to take advantage of the Trainee and data diagnostic results shown from the `engine-insights.ipynb` recipe. 

This can be done on a small scale where we show how a case can be edited or removed to modify the behavior of the Trainee. A Howso Engine session allows us to toggle entire batches of training data and add/remove large chunks of training data. This can be very useful if we are continuously adding data to our Trainee and we discover that certain batches are undesirable.

### Sessions

A Trainee Session is associated with each modification to a Trainee, which is useful for auditability. A session consists of the following information:  

- Unique identifier  

- The user for which the Session was created  

- Date the Session was created 

- Name, given by the user (Optional) 

- Metadata for the user to store information (Optional) 

When working with Trainees, a default session will be automatically started for you unless you explicitly start (or create) your own. This session will be used for all interactions with the Trainee, unless a new session is explicitly started, for as long as your client is running. Additionally, each instance of a Howso Client will use its own unique active session. Starting a new session explicitly is useful if you want to give it a name and/or metadata for your own reference later, or if you wish to use separate sessions for different modifications of the Trainee. For example, using a unique session each time you train would allow you to later reference the specific cases that were trained by a certain session. 


## Recipe Goals:

This notebook will show how to edit cases in a Howso Engine Trainee, either individually or in batches through the use of Sessions. This will allow the user to take actions on cases they deem necessary through use of the interpretability and auditing tools shown in other recipes.

In [1]:
import pandas as pd
from pmlb import fetch_data

from howso import engine
from howso.utilities import infer_feature_attributes

# Section 1: Train, Analyze, and Evaluate

For questions about the specific steps of this section, please see the [basic workflow guide](https://docs.howso.com/en/release-latest/user_guide/basic_capabilities/basic_workflow.html).

### Step 1: Load Data

Our example dataset for this recipe continues to be the well known `Adult` dataset. This dataset consists of 14 Context Features and 1 Action Feature. The Action Feature in this version of the `Adult` dataset has been renamed to `target` and it takes the form of a binary indicator for whether a person in the data makes over $50,000/year (*target*=0) or less (*target*=1).

In [2]:
df = fetch_data('adult', local_cache_dir="../../../data/pmlb")

# subsample the data to ensure the example runs quickly
df = df.sample(2000)

df

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,target
24219,45.0,6,34446.0,11,9.0,2,14,0,4,1,0.0,0.0,50.0,39,1
22456,32.0,4,387270.0,11,9.0,2,12,5,4,0,0.0,0.0,40.0,39,0
1564,42.0,1,141459.0,11,9.0,5,8,2,2,0,0.0,0.0,40.0,39,1
10497,24.0,4,49218.0,11,9.0,0,7,1,4,1,0.0,0.0,40.0,39,1
38643,57.0,0,179644.0,8,11.0,2,0,5,4,0,0.0,0.0,5.0,39,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45491,53.0,4,239155.0,15,10.0,2,4,0,4,1,0.0,0.0,50.0,39,0
7848,45.0,2,167159.0,9,13.0,2,11,0,4,1,0.0,0.0,50.0,39,0
4946,24.0,4,190290.0,9,13.0,4,12,1,4,1,0.0,0.0,60.0,39,1
17623,43.0,4,183765.0,15,10.0,0,13,1,4,1,0.0,0.0,40.0,0,0


### Step 2: Train Trainee utilizing Sessions

In this section we will perform all of the steps needed to train Howso Engine's Trainee.

In [3]:
# Infer features attributes
features = infer_feature_attributes(df, fanout_feature_map={"education": ["education-num"]})

# Specify Context and Action Features
action_features = ['target']
context_features = features.get_names(without=action_features)

# We extract one row for demonstrative purposes later
test_case = df.iloc[0]
df = df.iloc[1:]

# Split the data into Context Features (X) and Action Feature (y)
dfX = df[context_features]
dfy = df[action_features]

To demonstrate how to edit cases, we will break the training into two sessions. 

1. The first session is for half of the original dataset 
2. The second session is a modified version of the remaining half of the original dataset containing a target feature that is flipped from the true value

In [4]:
ind_session_1 = dfX.index[ :(dfX.shape[0] //2 )]
ind_session_2 = dfX.index[ (dfX.shape[0] //2 ):]

X_train_1 = dfX.loc[ind_session_1]
y_train_1 = dfy.loc[ind_session_1]

X_train_2 = dfX.loc[ind_session_2]

# Flip the target value for the second set of target feature values
y_train = dfy['target']
y_train_2 = pd.Series([int(not x) for x in y_train.loc[ind_session_2]], name=action_features[0], index=ind_session_2)

In [5]:
# Create the Trainee
trainee = engine.Trainee(
    features=features,
    overwrite_existing=True
)

session = engine.Session('train_session_1', metadata={'data': 'original data'})
trainee.train(X_train_1.join(y_train_1))

session = engine.Session('train_session_2', metadata={'data': 'modified data (flipped target values)'})
trainee.train(X_train_2.join(y_train_2))

# Analyze the Trainee
trainee.analyze()

✓ Train complete in 0:00:00


✓ Train complete in 0:00:00


✓ Analyze complete in 0:00:02


### Step 3: Inspect Results

In [6]:
accuracy = trainee.react_aggregate(
    details={
        "prediction_stats": True,
        "selected_prediction_stats": ['accuracy'],
    }
).to_dataframe()['target'].iloc[0]

print("Prediction accuracy: {acc}".format(acc=accuracy))

Prediction accuracy: 0.529


As expected, flipping the target feature's values for half of the data greatly reduces the accuracy compared to the expected accuracy shown in recipe `engine-intro.ipynb`.

While it is unrealistic to know this ahead of time in a real world setting, we will use this stark result to clearly demonstrate the effect of Trainee editing.


# Section 2: Audit & Edit a Trainee

There are many reasons to audit and edit a Trainee. In other recipes, we highlighted several training cases that may be anomalous that may be candidates for removal. In this recipe, we have an entire chunk of training data that is incorrect. Howso Engine has the ability to edit data at different scales.

What sets Howso Engine apart from other machine learning models is there is no need for retraining. For example, if we use `Scikit-Learn`'s Logistic regression and discover that our training data consists of cases we would like to remove, then we would have to go back to the beginning of the workflow, remove the problematic cases from the training data, and completely retrain the model.

In Howso Engine, this is unnecessary unless a very large portion of the training data is altered. If this is the case, re-analyzing the Trainee may be appropriate, although it is not strictly necessary.

### Step 1: Editing Individual Cases to Tune the Trainee

Here we demonstrate how to edit a Trainee one case at a time. Editing a case allows the user to modify or "fix" the behavior of the Trainee by the targeted editing of one or more cases. The user has complete control over all data in the Trainee, making it dynamic and quickly adjustable. Users do not need to worry about minor mistakes as the Trainee can be fine-tuned with this method after training.

In our use case, the anomalous cases identified for the `Adult` dataset in `anomaly_detection.ipynb` recipe represents possible cases we want to edit. We noticed certain cases with unusual values for `capital-gains` like 99999 that look like they are nominal values representing other values, such as blanks. Editing cases allows us to easily correct these minor issues for an otherwise valid case post-training. 

If we believe that a case is entirely invalid and warrants removal, Howso Engine can also remove it entirely.

To demonstrate this ability, we `react` to a single case to compare the predictions before and after an edit.

In [7]:
test_case_X = test_case[context_features]
test_case_y = test_case[action_features]

details = {
    'influential_cases':True,
}

new_result = trainee.react(
    [test_case_X.values.tolist()],
    context_features=context_features,
    action_features=action_features,
    details=details
)

✓ React complete in 0:00:00


In [8]:
print('prediction: {}'.format(int(new_result['action']['target'].iloc[0])))
print('actual: {}'.format(int(test_case_y.iloc[0])))

prediction: 0
actual: 1


#### Results 

Because half of the training data has flipped target values, this prediction may be incorrect. Whether or not it is, we can use our Trainee to push the prediction toward the actual value by editing its influential cases. This is for demonstrative purposes only and we do not recommend editing influential cases without fully investigating the cases.

### Step 2: Identify Influential Cases

To determine which cases we want to edit, we identify the Influential Cases.

In [9]:
influence_df = new_result['details']['influential_cases'][0]
influence_df

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,target,.session,.session_training_index,.influence_weight
0,51.0,4,33304.0,11,9.0,2,3,0,4,1,0.0,0.0,40.0,39,0,d4d0abc8-9e24-4539-82e0-2eb602ff43db,307,0.141076
1,43.0,6,32451.0,11,9.0,2,3,0,4,1,0.0,0.0,40.0,39,1,7a787992-b0f9-45b1-b933-9e74ea4e56d0,313,0.137459
2,40.0,4,30682.0,11,9.0,2,7,0,4,1,0.0,0.0,45.0,39,0,7a787992-b0f9-45b1-b933-9e74ea4e56d0,714,0.128279
3,29.0,4,37215.0,11,9.0,2,3,0,4,1,0.0,0.0,55.0,39,0,d4d0abc8-9e24-4539-82e0-2eb602ff43db,541,0.125170
4,42.0,4,29962.0,11,9.0,2,3,0,4,1,0.0,0.0,40.0,39,1,7a787992-b0f9-45b1-b933-9e74ea4e56d0,417,0.118450
5,50.0,6,29231.0,11,9.0,2,4,0,4,1,0.0,0.0,45.0,39,0,7a787992-b0f9-45b1-b933-9e74ea4e56d0,988,0.117633
6,49.0,6,39140.0,11,9.0,2,5,0,4,1,0.0,0.0,60.0,39,0,d4d0abc8-9e24-4539-82e0-2eb602ff43db,539,0.116514
7,44.0,1,38321.0,11,9.0,2,3,0,4,1,0.0,0.0,40.0,39,0,7a787992-b0f9-45b1-b933-9e74ea4e56d0,982,0.115418


We can see which Influential Cases have a target value that differs from the actual value of our test case.

### Step 3: Edit Cases

We will edit the top 3 Influential Cases so that their target value matches the value we want to predict. Any of them that already have that value are left unchanged. Having more Influential Cases with the correct target value will increase the chance of that case being predicted to the correct target value.

In [10]:
# Select the top 3 influential cases
session_ids = influence_df.iloc[[0, 1, 2]]['.session'].values.tolist()
session_training_indices = influence_df.iloc[[0, 1, 2]]['.session_training_index'].values.tolist()
case_indices = list(zip(session_ids, session_training_indices))

# The value we want the Trainee to predict for our test case
desired_value = int(test_case_y.iloc[0])

print(f"Editing cases: {case_indices}")
num_edited = trainee.edit_cases(
    feature_values=[[desired_value]],
    case_indices=case_indices,
    features=['target']
)
print(f"Edited {num_edited} cases")

Editing cases: [('d4d0abc8-9e24-4539-82e0-2eb602ff43db', 307), ('7a787992-b0f9-45b1-b933-9e74ea4e56d0', 313), ('7a787992-b0f9-45b1-b933-9e74ea4e56d0', 714)]
Edited 3 cases


### Step 4: Verify the Edit and Check the Case Audit

We can audit one of the updated cases to make sure the case has been edited and demonstrate how to retrieve the case history. Editing case history provides another layer of auditability and accountability to the Trainee.

In [11]:
updated_cases = trainee.get_cases(
    case_indices=case_indices,
    features=df.columns.tolist() + ['.case_edit_history']
)

# audit edit history
for case_edit_history in updated_cases[".case_edit_history"]:
    display(case_edit_history)

{'d4d0abc8-9e24-4539-82e0-2eb602ff43db': [{'feature': 'target',
   'type': 'edit',
   'value': 1,
   'previous_value': 0}]}

{'d4d0abc8-9e24-4539-82e0-2eb602ff43db': [{'feature': 'target',
   'type': 'edit',
   'value': 1,
   'previous_value': 1}]}

{'d4d0abc8-9e24-4539-82e0-2eb602ff43db': [{'feature': 'target',
   'type': 'edit',
   'value': 1,
   'previous_value': 0}]}

### Step 5: Predict Again

We will re-run the prediction to see if the target value is correct now.

In [12]:
new_result = trainee.react(
    [test_case_X.values.tolist()],
    context_features=context_features,
    action_features=action_features,
    details=details
)

print('prediction: {}'.format(int(new_result['action']['target'].iloc[0])))
print('actual: {}'.format(int(test_case_y.iloc[0])))

✓ React complete in 0:00:00


prediction: 1
actual: 1


By editing those three cases, we shifted the evidence behind this prediction toward the actual value without re-training or re-analyzing our Trainee. If the original prediction was incorrect, this is often enough to correct it. If done correctly, this provides a user with a surgical tool for Trainee corrections.

### Step 6: Delete a Case

In addition to editing a case, Howso Engine can also delete a case, removing it from the model and any further predictions. This workflow is the same as the edit example in the section above, except we use `remove_cases` instead of `edit_cases`. 

In [13]:
# remove cases using ".session_training_index"
trainee.remove_cases(num_cases=len(case_indices), case_indices=case_indices)

3

The dynamic editing and deleting of individual Cases allows the user to perform targeted modification of the Trainee and provides the user with unparalleled control over their data and Trainee. This should not be done lightly and we recommend that all Cases be investigated before performing this action.

# Section 3: Editing Sessions

In the beginning of this notebook, we trained the data in two sessions. The first session used a normal sample of training data, however the second session artificially flipped the target variable. This reduced the performance of our Trainee by introducing a large portion of incorrect data.

Howso Engine has the capability to add or remove entire sessions. In this situation, if we discovered that one of our sessions had very poor quality data, like our example, we can easily remove that entire session's data without having to individually alter cases.

### Step 1: View Sessions

Let's first see how many sessions are in this Trainee along with some details of each session.

In [14]:
sessions = trainee.get_sessions()
sessions

[{'id': '7a787992-b0f9-45b1-b933-9e74ea4e56d0', 'name': 'train_session_1'},
 {'id': 'd4d0abc8-9e24-4539-82e0-2eb602ff43db', 'name': 'train_session_2'}]

In [15]:
display(engine.get_session(sessions[0]['id']))
display(engine.get_session(sessions[1]['id']))

{'id': '7a787992-b0f9-45b1-b933-9e74ea4e56d0',
 'name': 'train_session_1',
 'metadata': {'data': 'original data',
              'trainee_id': 'aa6054ff-a768-4bc5-92fb-8cceff61f6ad'},
 'created_date': datetime.datetime(2026, 11, 23, 18, 19, 43, 661778, tzinfo=tzlocal()),
 'modified_date': datetime.datetime(2026, 11, 23, 18, 19, 43, 661783, tzinfo=tzlocal())}

{'id': 'd4d0abc8-9e24-4539-82e0-2eb602ff43db',
 'name': 'train_session_2',
 'metadata': {'data': 'modified data (flipped target values)'},
 'created_date': datetime.datetime(2026, 11, 23, 18, 19, 43, 688168, tzinfo=datetime.timezone.utc),
 'modified_date': datetime.datetime(2026, 11, 23, 18, 19, 43, 688169, tzinfo=datetime.timezone.utc)}

We can see the two different sessions we used when trained earlier.

### Step 2: Delete a Session

Deleting an entire session is performed in one easy step once we retrieve the session ID of the session we want to delete.

In [16]:
# Delete a session
session_ids = sessions[1]['id']
trainee.delete_session(session_ids)

# Re-analyze the Trainee
trainee.analyze()

✓ Analyze complete in 0:00:02


### Step 3: Recompute Accuracy and Inspect

We then use `react_aggregate` to compute the accuracy metrics.

In [17]:
accuracy_new = trainee.react_aggregate(
    details={
        "prediction_stats": True,
        "selected_prediction_stats": ['accuracy'],
    }
).to_dataframe()['target'].iloc[0]

print("Original accuracy: {acc}".format(acc=accuracy))
print("New accuracy: {acc}".format(acc=accuracy_new))

Original accuracy: 0.529
New accuracy: 0.8405215646940822


In [18]:
# Check to make sure there is only 1 session
trainee.get_sessions()

[{'id': '7a787992-b0f9-45b1-b933-9e74ea4e56d0', 'name': 'train_session_1'}]

We can clearly see the difference in accuracy results once the faulty session data is removed.

Additionally, we see only one session remains in the Trainee.

# Conclusion:

We can see that by getting rid of the session with the faulty data, our Trainee performance improved dramatically, as expected. This capability provides the user with a very efficient way to maintain control over a continuously evolving Trainee if the user is constantly adding training data.

The tools shown in this and other recipes allows the user to find, diagnose, and act at a level of ease and precision that other machine learning models cannot match. This opens the door to possibilities for the user and provides a flexible platform that can adjust to any type of machine learning needs.